# Environment Setup

In [12]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [11]:
# !pip install transformers

In [10]:
# !pip install pandas

In [7]:
# !pip install --no-cache-dir "pyarrow>=14.0.1,<18"

In [8]:
# !pip install tqdm

In [9]:
# !pip install requests

# RELAION-5B Download

In [ ]:
import os

# 1. Read Hugging Face token from file
with open("hf_token.txt", "r", encoding="utf-8") as f:
    hf_token = f.read().strip()

# 2. Set environment variable
os.environ["HF_TOKEN"] = hf_token

# 3. Use it (requests, datasets, etc.)
assert "HF_TOKEN" in os.environ

In [ ]:
import os
import requests
import subprocess
from typing import List

def get_relaion_parquet_urls(dataset_name: str, hf_token: str) -> List[str]:
    """
    Query Hugging Face datasets-server API and return all parquet shard URLs
    for the given dataset.
    """
    api_url = "https://datasets-server.huggingface.co/parquet"
    headers = {
        "Authorization": f"Bearer {hf_token}"
    }
    params = {
        "dataset": dataset_name
    }

    resp = requests.get(api_url, headers=headers, params=params)
    resp.raise_for_status()
    data = resp.json()

    parquet_urls = []

    for pf in data.get("parquet_files", []):
        url = pf.get("url")
        if url:
            parquet_urls.append(url)

    if not parquet_urls:
        raise RuntimeError("No parquet files found. Check dataset access or schema.")

    return parquet_urls

def download_parquets(
    dataset_name: str,
    n: int,
    output_dir: str,
    hf_token: str,
):
    """
    Download the first n parquet files of a Hugging Face dataset
    into output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    parquet_urls = get_relaion_parquet_urls(dataset_name, hf_token)

    print(f"[INFO] Found {len(parquet_urls)} parquet shards total")
    print(f"[INFO] Downloading first {n} shards → {output_dir}")

    for i, url in enumerate(parquet_urls[:n]):
        out_path = os.path.join(output_dir, f"part_{i:04d}.parquet")

        print(f"[INFO] ({i+1}/{n}) Downloading {out_path}")

        subprocess.run(
            [
                "curl",
                "-L",
                "-H", f"Authorization: Bearer {hf_token}",
                "-H", "Accept: application/octet-stream",
                "-o", out_path,
                url,
            ],
            check=True,
        )

    print("[INFO] Download complete")

DATASET = "laion/relaion2B-en-research-safe"
N_PARQUETS = 4
OUTPUT_DIR = "F:\Thesis\RE-LAION-5B_Dataset"

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    raise RuntimeError("HF_TOKEN environment variable not set")

download_parquets(
    dataset_name=DATASET,
    n=N_PARQUETS,
    output_dir=OUTPUT_DIR,
    hf_token=HF_TOKEN,
)